Start by converting CSV files into dataframes.

In [46]:
import pandas as pd

# Convert CSV files into dataframes
orders = pd.read_csv('../dataset/orders.csv')
order_products_prior = pd.read_csv('../dataset/order_products__prior.csv')
products = pd.read_csv('../dataset/products.csv')
aisles = pd.read_csv('../dataset/aisles.csv')
departments = pd.read_csv('../dataset/departments.csv')

Merge all into a single flat dataframe.

In [47]:
# Merge products df with department and aisle details
products_full = (products
    .merge(aisles, on='aisle_id', how='left')
    .merge(departments, on='department_id', how='left')
)

# Check if all aisles belong to a single department (aisle is subgroup of department)
aisle_department_counts = products_full.groupby('aisle_id')['department_id'].nunique()

# Check if any aisle belongs to more than one department
if (aisle_department_counts > 1).any():
    print("Some aisles belong to multiple departments")
else:
    print("All aisles belong to a single department")


# Merge products df with historical orders df
prior_products = order_products_prior.merge(products_full, on='product_id', how='left')

# Merge with order info
full_orders_products = prior_products.merge(
    orders,
    on='order_id',
    how='left'
)

# Convert object dtypes for efficient manipulation
full_orders_products['product_name'] = full_orders_products['product_name'].astype('string')
full_orders_products['department'] = full_orders_products['department'].astype('category')
full_orders_products['aisle'] = full_orders_products['aisle'].astype('category')

All aisles belong to a single department
   order_id  product_id  add_to_cart_order  reordered           product_name  \
0         2       33120                  1          1     Organic Egg Whites   
1         2       28985                  2          1  Michigan Organic Kale   
2         2        9327                  3          0          Garlic Powder   
3         2       45918                  4          1         Coconut Butter   
4         2       30035                  5          0      Natural Sweetener   

   aisle_id  department_id               aisle  department  user_id eval_set  \
0        86             16                eggs  dairy eggs   202279    prior   
1        83              4    fresh vegetables     produce   202279    prior   
2       104             13   spices seasonings      pantry   202279    prior   
3        19             13       oils vinegars      pantry   202279    prior   
4        17             13  baking ingredients      pantry   202279    prior  

Check for duplicates.

In [4]:
# Checking if there are instances where a product appears in an order multiple times
duplicates_within_orders = (
    full_orders_products
    .duplicated(subset=['order_id', 'product_name'], keep=False)
)
print("Number of (order_id, product) duplicates:",
      duplicates_within_orders.sum())

Number of (order_id, product) duplicates: 0


Clean up unnecessary columns.

In [21]:
# Remove columns that are not needed after merging
columns_to_remove = ["add_to_cart_order","reordered", "order_dow", "order_hour_of_day", "days_since_prior_order", "eval_set"]

truncated_df = full_orders_products.drop(columns=columns_to_remove)

truncated_df.head()

truncated_df.to_csv('../data/cleaned/order-products-full.csv', index=False)

ORDER ANALYSIS

Order Size

In [49]:
order_info_df = orders[orders['eval_set'] == 'prior']
order_info_df = order_info_df[['order_id', 'user_id']]

# Calculate order size
order_sizes = truncated_df.groupby('order_id')['product_id'].count()
order_sizes = order_sizes.reset_index(name='order_size')
order_size_counts = order_sizes['order_size'].value_counts().sort_index()

# Statistics for order size
print(order_sizes['order_size'].describe())

# Distribution for order sizes
plt.bar(order_size_counts.index, order_size_counts.values, color='skyblue', edgecolor='black')
plt.title("Order Size Distribution")
plt.xlabel("Number of Products per Order")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(f"../graphs/order_size_frequency.png") 
plt.close()

# Build dataframe for analysis
order_info_full_df = order_info_df.copy()
order_info_full_df = order_info_full_df.merge(order_sizes, on='order_id', how='left')

count    3.214874e+06
mean     1.008888e+01
std      7.525398e+00
min      1.000000e+00
25%      5.000000e+00
50%      8.000000e+00
75%      1.400000e+01
max      1.450000e+02
Name: order_size, dtype: float64


Orders per User

In [50]:
# Count number of orders per user
user_order_counts = order_info_df.groupby('user_id')['order_id'].nunique()
user_order_counts = user_order_counts.reset_index(name='user_order_count')

# Statistics for orders per user
print(user_order_counts['user_order_count'].describe())

# Distribution for orders per user
plt.hist(user_order_counts['user_order_count'], bins=50, color='skyblue', edgecolor='black')
plt.xlabel("Number of Orders per User")
plt.ylabel("Number of Users")
plt.title("Distribution of Orders per User")
plt.savefig(f"../graphs/user_order_counts_hist.png") 
plt.close()

# Add orders per user to exisiting df
order_info_full_df = order_info_full_df.merge(user_order_counts, on='user_id', how='left')

count    206209.000000
mean         15.590367
std          16.654774
min           3.000000
25%           5.000000
50%           9.000000
75%          19.000000
max          99.000000
Name: user_order_count, dtype: float64


Products per User

In [52]:
# Count number of unique products purchased per user
user_product_counts = (
    truncated_df.groupby('user_id')['product_id']
    .nunique()
    .reset_index(name='user_product_count')
)

# Statistics for products per user
print(user_product_counts['user_product_count'].describe())

# Distribution for products per user
plt.hist(user_product_counts['user_product_count'], bins=50, color='skyblue', edgecolor='black')
plt.xlabel("Number of Products per User")
plt.ylabel("Number of Users")
plt.title("Distribution of Products per User")
plt.savefig(f"../graphs/user_products_counts_hist.png") 
plt.close()

# Add products per user to exisiting df
order_info_full_df = order_info_full_df.merge(user_product_counts, on='user_id', how='left')

# Create summary and save
order_summary = order_info_full_df.describe()
order_summary.to_csv('../data/summary/order-info-summary.csv', index=False)

# Save full order info
order_info_full_df.to_csv('../data/cleaned/order-info-full.csv', index=False)

count    206209.000000
mean         64.536238
std          56.592339
min           1.000000
25%          25.000000
50%          48.000000
75%          86.000000
max         726.000000
Name: user_product_count, dtype: float64


PRODUCT ANALYSIS

Orders per Product

In [54]:
product_info_df = products.copy()
product_info_full_df = products.copy()

# Calculate number of orders a product has appeared in
total_orders = truncated_df['order_id'].nunique()
product_counts = truncated_df['product_id'].value_counts().reset_index()
product_counts.columns = ['product_id', 'count']

# Add column to show percentage of orders containing each product
product_counts['order_penetration_pct'] = (product_counts['count'] / total_orders) * 100

# Show statistics
print(product_counts['count'].describe())
print(product_counts['order_penetration_pct'].describe())

# Distribution for orders per product
plt.figure(figsize=(8, 5))
plt.hist(product_counts['count'], bins=50, color='skyblue', edgecolor='black')
plt.title("Distribution of Product Counts")
plt.xlabel("Number of Orders")
plt.ylabel("Number of Products")
plt.tight_layout()
plt.savefig(f"../graphs/product_order_counts_hist2.png") 
plt.close()

# Add columns for analysis
product_info_full_df = product_info_full_df.merge(product_counts, on='product_id', how='left')

count     49677.000000
mean        652.907563
std        4792.114416
min           1.000000
25%          17.000000
50%          60.000000
75%         260.000000
max      472565.000000
Name: count, dtype: float64
count    49677.000000
mean         0.020309
std          0.149061
min          0.000031
25%          0.000529
50%          0.001866
75%          0.008087
max         14.699332
Name: order_penetration_pct, dtype: float64


Users per product.

In [56]:
# Calculate number of users that have purchased the product
total_users = truncated_df['user_id'].nunique()

# Count number of unique users that purchased each product
user_counts = (
    truncated_df.groupby('product_id')['user_id']
    .nunique()
    .reset_index(name='user_count')
)

# Add column for percentage of users who purchased each product (user penetration)
user_counts['user_penetration_pct'] = (user_counts['user_count'] / total_users) * 100

# Distribution for users per product
plt.figure(figsize=(8, 5))
plt.hist(user_counts['user_count'], bins=50, color='skyblue', edgecolor='black')
plt.title("Distribution of User Counts")
plt.xlabel("Number of Users")
plt.ylabel("Number of Products")
plt.tight_layout()
plt.savefig(f"../graphs/product_user_counts_hist.png") 
plt.close()

# Display summary statistics
print(user_counts['user_count'].describe())
print(user_counts['user_penetration_pct'].describe())

# Add columns for analysis
product_info_full_df = product_info_full_df.merge(user_counts, on='product_id', how='left')

count    49677.000000
mean       267.889627
std       1308.788623
min          1.000000
25%         11.000000
50%         35.000000
75%        137.000000
max      73956.000000
Name: user_count, dtype: float64
count    49677.000000
mean         0.129912
std          0.634690
min          0.000485
25%          0.005334
50%          0.016973
75%          0.066437
max         35.864584
Name: user_penetration_pct, dtype: float64


Products per Department.

In [68]:
# Number of products in each department
products_per_department = (
    product_info_df.groupby('department_id')['product_id']
    .nunique()
    .reset_index(name='dept_size')
)

# Show statistics
print(products_per_department['dept_size'].describe())

# Distribution for department sizes
plt.bar(products_per_department['dept_size'].index, products_per_department['dept_size'].values, color='skyblue', edgecolor='black')
plt.title("Department Size Distribution")
plt.xlabel("Number of Departments")
plt.ylabel("Number of Products")
plt.tight_layout()
plt.savefig(f"../graphs/dept_size_frequency.png") 
plt.close()

# Number of unique aisles in each department
aisles_per_department = (
    product_info_df.groupby('department_id')['aisle_id']
    .nunique()
    .reset_index(name='num_aisles')
)

# Show statistics
print(aisles_per_department['num_aisles'].describe())

# Add columns for analysis
product_info_full_df = product_info_full_df.merge(products_per_department, on='department_id', how='left')

count      21.000000
mean     2366.095238
std      1914.367491
min        38.000000
25%      1081.000000
50%      1516.000000
75%      3449.000000
max      6563.000000
Name: dept_size, dtype: float64
count    21.000000
mean      6.380952
std       4.128876
min       1.000000
25%       4.000000
50%       5.000000
75%      10.000000
max      17.000000
Name: num_aisles, dtype: float64


Orders per department.

In [13]:
import matplotlib.pyplot as plt

# Calculate number of orders a department has appeared in
total_orders = truncated_df['order_id'].nunique()
department_counts = (
    truncated_df[['order_id', 'department_id']]
    .drop_duplicates() 
    .groupby('department_id')['order_id']
    .nunique()
    .reset_index()
    .rename(columns={'order_id': 'count'})
)

# Add column to show percentage of orders containing each department
department_counts['dept_penetration_pct'] = (department_counts['count'] / total_orders) * 100

# Display summary statistics
print(department_counts['count'].describe())
print(department_counts['dept_penetration_pct'].describe())

# Add columns for analysis
product_info_full_df = product_info_full_df.merge(department_counts, on='department_id', how='left')

count    2.100000e+01
mean     7.250570e+05
std      6.866174e+05
min      3.380200e+04
25%      1.777120e+05
50%      5.747310e+05
75%      1.117892e+06
max      2.409320e+06
Name: count, dtype: float64
count    21.000000
mean     22.553203
std      21.357520
min       1.051425
25%       5.527806
50%      17.877248
75%      34.772498
max      74.942906
Name: dept_penetration_pct, dtype: float64


Products per Aisle.

In [67]:
# Number of products in each aisle
products_per_aisle = (
    product_info_df.groupby('aisle_id')['product_id']
    .nunique()
    .reset_index(name='aisle_size')
)

# Show statistics
print(products_per_aisle['aisle_size'].describe())

# Distribution for aisle sizes
plt.bar(products_per_aisle['aisle_size'].index, products_per_aisle['aisle_size'].values, color='skyblue', edgecolor='black')
plt.title("Aisle Size Distribution")
plt.xlabel("Number of Aisles")
plt.ylabel("Number of Products")
plt.tight_layout()
plt.savefig(f"../graphs/aisle_size_frequency.png") 
plt.close()

# Add columns for analysis
product_info_full_df = product_info_full_df.merge(products_per_aisle, on='aisle_id', how='left')

count     134.000000
mean      370.805970
std       267.010165
min        12.000000
25%       179.750000
50%       305.500000
75%       497.500000
max      1258.000000
Name: aisle_size, dtype: float64


Orders per Aisle.

In [15]:
# Calculate number of orders an aisle has appeared in
total_orders = truncated_df['order_id'].nunique()
aisle_counts = (
    truncated_df[['order_id', 'aisle_id']]
    .drop_duplicates()  
    .groupby('aisle_id')['order_id']
    .nunique()
    .reset_index()
    .rename(columns={'order_id': 'count'})
)

# Add column to show percentage of orders containing each aisle
aisle_counts['aisle_penetration_pct'] = (aisle_counts['count'] / total_orders) * 100

# Display summary statistics
print(aisle_counts['count'].describe())
print(aisle_counts['aisle_penetration_pct'].describe())

# Add columns for analysis
product_info_full_df = product_info_full_df.merge(aisle_counts, on='aisle_id', how='left')

# Create summary and save
product_summary = product_info_full_df.describe()
product_summary.to_csv('../data/summary/product-info-summary.csv', index=False)

# Save the dataframe for future use
product_info_full_df.to_csv('../data/cleaned/product-info-full.csv', index=False)


count    1.340000e+02
mean     1.741676e+05
std      2.573764e+05
min      4.239000e+03
25%      2.681150e+04
50%      8.730750e+04
75%      2.238022e+05
max      1.790771e+06
Name: count, dtype: float64
count    134.000000
mean       5.417555
std        8.005799
min        0.131856
25%        0.833983
50%        2.715736
75%        6.961463
max       55.702681
Name: aisle_penetration_pct, dtype: float64


EDA Report

In [16]:
from ydata_profiling import ProfileReport


# Build a sample of users to keep order integrity and product diversity
sample_frac = 0.05
sampled_users = orders['user_id'].drop_duplicates().sample(frac=sample_frac, random_state=42)
df_sample = full_orders_products[full_orders_products['user_id'].isin(sampled_users)]

profile = ProfileReport(df_sample, explorative=True)
profile.to_file("eda_report_orders.html")

c:\Users\jenle\AppData\Local\Programs\Python\Python312\Lib\site-packages\ydata_profiling\utils\dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"index": "df_index"}, inplace=True)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 15/15 [00:08<00:00,  1.83it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Build distribution graphs.

In [ ]:
%run ../utils/data-preparation.py


# Order size
order_size_counts = order_info_full_df['order_size'].value_counts().sort_index()
build_bar_chart(order_size_counts, "order", "product")

# Orders per user
user_order_counts = order_info_full_df['user_order_count'].value_counts().sort_index()
build_bar_chart(user_order_counts, "order", "user")

# Products per user
user_product_counts = order_info_full_df['user_product_count'].value_counts().sort_index()
build_bar_chart(user_product_counts, "product", "user")

# Orders per product
product_orders_counts = product_info_full_df['count_x'].value_counts().sort_index()
build_hist_distribution(product_orders_counts, "product", "order")

ADDITIONAL DATA PREPARATION

Build and save product pair probabilities.

In [10]:
%run ../utils/pairwise.py
%run ../utils/sampling.py

products = pd.read_csv('../data/cleaned/product-info-full.csv')
#product_counts = products.rename(columns={ 'count_x': 'count'})

# Get sample of products
sampled_products = product_sampling_fixed_size(
    products,
    target_sample_size=5000,
    min_orders=50
)

# Convert list to dataframe and save as CSV
sampled_products_df = pd.DataFrame(sampled_products, columns=['product_id'])
sampled_products_df.to_csv('../data/validation/sampled-products.csv', index=False)

Sampled 5000 products from 49688 eligible products.


In [31]:
products = pd.read_csv('../dataset/products.csv')

aisles = products[products['product_id'].isin(sampled_products_df['product_id'])]
aisle_ids = departments['aisle_id'].unique()
print(sorted(aisle_ids))


[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134]


In [29]:
%run ../utils/pairwise.py
import pandas as pd

order_products_prior = pd.read_csv('../dataset/order_products__prior.csv')
products_df = pd.read_csv('../dataset/products.csv')

# Reduce dataframe to two necessary columns
order_product_df = order_products_prior[['order_id', 'product_id']]

dept_id = 21
filtered_products = products_df[products_df['department_id'] == dept_id]['product_id']

# If you want it as a list
product_list = filtered_products.tolist()

# Compute pairwise probabilties to be used for later calculations
compute_pairwise_probabilities_sample(order_product_df, 
    product_list,
    output_csv="../data/cleaned/pairwise-dept21.csv",
    batch_size=1000
)

100%|██████████| 2/2 [00:08<00:00,  4.29s/it]

Completed computation. Saved to ../data/cleaned/pairwise-dept21.csv


In [30]:
%run ../utils/pairwise.py
%run ../utils/sampling.py

order_products_prior = pd.read_csv('../dataset/order_products__prior.csv')
sampled_products_df = pd.read_csv('../data/validation/sampled-products.csv')
order_product_df = order_products_prior[['order_id', 'product_id']]

# Compute pairwise probabilties to be used for later calculations
product_pair_df = compute_pairwise_probabilities_sample(order_product_df, 
    sampled_products_df['product_id'].tolist(),
    output_csv="../data/validation/sample-pairwise.csv",
    batch_size=1000
)

100%|██████████| 5/5 [02:27<00:00, 29.45s/it] 


Completed computation. Saved to ../data/validation/sample-pairwise.csv


Normalize features for order and product dataframes

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Normalize features related to absolute size
order_info_full_df["order_size_scaled"] = MinMaxScaler().fit_transform(order_info_full_df[["order_size"]])
order_info_full_df["orders_per_user_scaled"] = MinMaxScaler().fit_transform(order_info_full_df[["user_order_count"]])
order_info_full_df["prod_per_user_scaled"] = MinMaxScaler().fit_transform(order_info_full_df[["user_product_count"]])
product_info_full_df["dept_size_scaled"] = MinMaxScaler().fit_transform(product_info_full_df[["dept_size_x"]])
product_info_full_df["aisle_size_scaled"] = MinMaxScaler().fit_transform(product_info_full_df[["aisle_size"]])

# Remove columns no longer needed
product_cols_to_remove = ["product_name", "count_x", "user_count", "dept_size_x", "dept_size_y", "count_y", "aisle_size", "count"]
order_cols_to_remove = ["order_size", "user_order_count", "user_product_count"]

# Apply to products
product_feature_df = product_info_full_df.drop(columns=product_cols_to_remove)

product_feature_df.to_csv('../data/cleaned/product-features.csv', index=False)

# Repeat for orders
order_feature_df = order_info_full_df.drop(columns=order_cols_to_remove)

order_feature_df.to_csv('../data/cleaned/order-features.csv', index=False)


In [ ]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer, util
from rapidfuzz import fuzz
import numpy as np
from itertools import combinations

df = pd.read_csv('../dataset/products.csv') 

# Normalize product names
def normalize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['norm_name'] = df['product_name'].apply(normalize)


# Initialize sentence-transformers model
model = SentenceTransformer('all-MiniLM-L6-v2')


def compute_similarity(df, fuzzy_threshold=70, cosine_threshold=0.7, batch_size=500):
    similar_pairs = []

    # Process by department to reduce comparisons
    for aisle_id, aisle_group in df.groupby('aisle_id'):
        products = aisle_group['norm_name'].tolist()
        original_names = aisle_group['product_name'].tolist()
        product_ids = aisle_group['product_id'].tolist()
        indices = aisle_group.index.tolist()

        # Fuzzy pre-filter
        """ candidate_pairs = []
        for i, j in combinations(range(len(products)), 2):
            score = fuzz.token_sort_ratio(products[i], products[j])
            if score >= fuzzy_threshold:
                candidate_pairs.append((i, j))

        if not candidate_pairs:
            continue """
        
        all_pairs = list(combinations(range(len(products)), 2))

        # Compute embeddings for this department in batch
        embeddings = model.encode(products, convert_to_tensor=False, batch_size=batch_size)
        embeddings = np.vstack(embeddings).astype('float32')
        # Normalize for cosine similarity
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        embeddings = embeddings / norms

        # Compute cosine similarity for candidate pairs
        for i, j in all_pairs:
            cos_sim = float(np.dot(embeddings[i], embeddings[j]))
            if cos_sim >= cosine_threshold:
                # Combine fuzzy + cosine into a single score (optional)
                combined_score = round((cos_sim * 0.6), 3)  # weighted
                similar_pairs.append({
                    "product_i": product_ids[i],   
                    "product_j": product_ids[j], 
                    "product_i_name": original_names[i],
                    "product_j_name": original_names[j],
                    "cosine": round(cos_sim,3),
                    "combined": combined_score
                })
    return pd.DataFrame(similar_pairs)


similar_df = compute_similarity(df, fuzzy_threshold=70, cosine_threshold=0.7)
similar_df.to_csv('../data/cleaned/product-similiarity.csv', index=False)
print(similar_df.head(20))


    product_i  product_j            product_i_name  \
0         209      22853       Italian Pasta Salad   
1         209      27216       Italian Pasta Salad   
2         209      33016       Italian Pasta Salad   
3         209      42161       Italian Pasta Salad   
4         209      44705       Italian Pasta Salad   
5         209      46897       Italian Pasta Salad   
6         554      18864              Turkey Chili   
7        1600      21384  Mediterranean Orzo Salad   
8        1600      28008  Mediterranean Orzo Salad   
9        2539       4369     Original Potato Salad   
10       2539       9431     Original Potato Salad   
11       2539      13735     Original Potato Salad   
12       2539      16618     Original Potato Salad   
13       2539      22182     Original Potato Salad   
14       2539      29746     Original Potato Salad   
15       2539      33878     Original Potato Salad   
16       2539      37167     Original Potato Salad   
17       2539      41520    